In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import dotenv_values

In [3]:
env = dotenv_values(".env")

In [4]:
engine = create_engine(env["POSTGRES_URI"], pool_pre_ping=True, pool_recycle=900, pool_timeout=30)


def check_postgres_connection():
    try:
        with engine.connect() as connection:
            result = connection.execute(text("SELECT 'Connected Successfully'"))
            return result.fetchone()[0]
    except Exception as e:
        return f"Error connecting to PostgreSQL: {e}"

check_postgres_connection()

'Connected Successfully'

In [5]:
display(pd.read_sql("SELECT COUNT(*) from dataengineering_project_customers", engine))
display(pd.read_sql("SELECT COUNT(*) from dataengineering_project_orders", engine))

,count
0,2000


,count
0,200000


In [6]:
customers_df = pd.read_sql("SELECT * from dataengineering_project_customers", engine)

In [7]:
customers_df.head(5)

,customer_id,name,city,segment,signup_date
0,1,Usman Sheikh,Karachi,Consumer,2022-11-28
1,2,Iqra Khan,Faisalabad,Consumer,2022-11-11
2,3,Maryam Raza,Faisalabad,Consumer,2023-10-20
3,4,Ali Sheikh,Lahore,Consumer,2022-08-09
4,5,Hamza Qureshi,Karachi,Consumer,2022-11-10


In [8]:
display(customers_df.shape)
print("\n", "--"*50, "\n")
display(customers_df.columns)
print("\n", "--"*50, "\n")
display(customers_df.info())
print("\n", "--"*50, "\n")
display(customers_df.describe())

(2000, 5)


 ---------------------------------------------------------------------------------------------------- 



Index(['customer_id', 'name', 'city', 'segment', 'signup_date'], dtype='str')


 ---------------------------------------------------------------------------------------------------- 

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  2000 non-null   int64
 1   name         2000 non-null   str  
 2   city         2000 non-null   str  
 3   segment      2000 non-null   str  
 4   signup_date  2000 non-null   str  
dtypes: int64(1), str(4)
memory usage: 78.3 KB


None


 ---------------------------------------------------------------------------------------------------- 



,customer_id
count,2000.000000
mean,1000.500000
std,577.494589
min,1.000000
25%,500.750000
50%,1000.500000
75%,1500.250000
max,2000.000000


In [9]:
orders_df = []

In [10]:
orders_counts = pd.read_sql("SELECT COUNT(*) from dataengineering_project_orders", engine)
print(orders_counts)
# orders_df = pd.read_sql("SELECT * from dataengineering_project_orders", engine)

    count
0  200000


In [11]:
from tqdm.auto import tqdm

orders, rows = [], 0

for chunk in tqdm(
    pd.read_sql(
        "SELECT * FROM dataengineering_project_orders",
        engine,
        chunksize=50_000
    ),
    desc="Loading Orders",
    unit="chunk"
):
    rows += len(chunk)
    orders.extend(chunk.to_dict("records"))

print(f"Total rows loaded: {rows:,}")

/Volumes/Coding Work Place/Data Engineer Project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading Orders: 4chunk [00:08,  2.18s/chunk]

Total rows loaded: 200,000


In [12]:
len(orders)

200000

In [13]:
import certifi
from pymongo import MongoClient

try:
    client = MongoClient( env["MONGO_URI"], serverSelectionTimeoutMS=5000, tlsCAFile=certifi.where() )
    
    client.admin.command('ping')
    print("Successfully connected to MongoDB!")

    db = client["dataengineering_db"]
    orders_df = pd.DataFrame(list(db["orders"].find({})))

    print(orders_df.head())
    print(f"Shape: {orders_df.shape}")

except Exception as e:
    print(f"Connection failed: {e}")

Successfully connected to MongoDB!
                        _id  order_id           order_date     status  \
0  6ae8c4a71dc54dd2c64a613b    287309  2025-01-02T04:28:40  cancelled   
1  6ae8c4a71dc54dd2c64a6151    133843  2025-01-04T01:23:47    pending   
2  6ae8c4a71dc54dd2c64a615d    126946  2025-01-05T18:26:56  delivered   
3  6ae8c4a71dc54dd2c64a615e    162981  2025-01-05T22:55:02  delivered   
4  6ae8c4a71dc54dd2c64a6163    294333  2025-01-06T06:28:47  delivered   

     amount                                         items  \
0  13034.40  [{'category': 'Electronics', 'quantity': 1}]   
1  10997.53  [{'category': 'Electronics', 'quantity': 1}]   
2  42176.02  [{'category': 'Electronics', 'quantity': 3}]   
3    446.09        [{'category': 'Books', 'quantity': 1}]   
4   5856.50      [{'category': 'Grocery', 'quantity': 1}]   

                          payment          updated_at  \
0              {'method': 'card'} 2025-01-03 06:49:40   
1  {'method': 'cash_on_delivery'} 2025-01-04 

In [14]:
customers_df.head()

,customer_id,name,city,segment,signup_date
0,1,Usman Sheikh,Karachi,Consumer,2022-11-28
1,2,Iqra Khan,Faisalabad,Consumer,2022-11-11
2,3,Maryam Raza,Faisalabad,Consumer,2023-10-20
3,4,Ali Sheikh,Lahore,Consumer,2022-08-09
4,5,Hamza Qureshi,Karachi,Consumer,2022-11-10


In [15]:
orders_df.head()

,_id,order_id,order_date,status,amount,items,payment,updated_at,customer,customer_id,coupon,delivery_notes
0,6ae8c4a71dc54dd2c64a613b,287309,2025-01-02T04:28:40,cancelled,13034.40,"[{'category': 'Electronics', 'quantity': 1}]",{'method': 'card'},2025-01-03 06:49:40,None,1553.0,ELEVEN11,NaN
1,6ae8c4a71dc54dd2c64a6151,133843,2025-01-04T01:23:47,pending,10997.53,"[{'category': 'Electronics', 'quantity': 1}]",{'method': 'cash_on_delivery'},2025-01-04 10:09:47,None,496.0,NaN,NaN
2,6ae8c4a71dc54dd2c64a615d,126946,2025-01-05T18:26:56,delivered,42176.02,"[{'category': 'Electronics', 'quantity': 3}]",{'method': 'bank_transfer'},2025-01-09 00:40:56,"{'id': 1920, 'name': 'Hassan Mirza', 'city': '...",NaN,NaN,NaN
3,6ae8c4a71dc54dd2c64a615e,162981,2025-01-05T22:55:02,delivered,446.09,"[{'category': 'Books', 'quantity': 1}]",{'method': 'card'},2025-01-11 04:22:02,"{'id': 43, 'name': 'Saad Siddiqui', 'city': 'L...",NaN,NaN,NaN
4,6ae8c4a71dc54dd2c64a6163,294333,2025-01-06T06:28:47,delivered,5856.50,"[{'category': 'Grocery', 'quantity': 1}]",{'method': 'cash_on_delivery'},2025-01-09 11:28:47,None,1176.0,NaN,NaN


In [19]:
pg_orders_df = pd.DataFrame(orders)

In [20]:
pg_orders_df

,order_id,customer_id,order_date,category,quantity,amount,status,payment_method,updated_at
0,256821,1419,2025-01-01,Beauty,1,575.98,pending,wallet,2025-01-01 05:34:09
1,208443,701,2025-01-01,Fashion,2,5685.46,delivered,bank_transfer,2025-01-05 08:11:23
2,265974,430,2025-01-01,Electronics,1,35507.97,delivered,wallet,2025-01-03 06:08:32
3,221969,307,2025-01-01,Beauty,1,1383.10,delivered,bank_transfer,2025-01-08 06:29:24
4,271915,541,2025-01-01,Home,3,69759.11,delivered,wallet,2025-01-03 07:45:02
...,...,...,...,...,...,...,...,...,...
199995,174217,1486,2026-10-24,Beauty,2,13803.09,delivered,card,2026-10-24 23:59:00
199996,281914,1243,2026-10-24,Grocery,1,3533.85,delivered,card,2026-10-24 23:59:00
199997,188212,1149,2026-10-24,Books,1,2375.13,pending,bank_transfer,2026-10-24 23:59:00
199998,171064,942,2026-10-24,Electronics,1,47674.64,delivered,wallet,2026-10-24 23:59:00


In [22]:
from pathlib import Path
Path("raw").mkdir(parents=True, exist_ok=True)

In [24]:
customers_df.to_csv("raw/customers.csv", index=False)
pg_orders_df.to_csv("raw/orders.csv", index=False)

In [25]:
orders_df.head()

,_id,order_id,order_date,status,amount,items,payment,updated_at,customer,customer_id,coupon,delivery_notes
0,6ae8c4a71dc54dd2c64a613b,287309,2025-01-02T04:28:40,cancelled,13034.40,"[{'category': 'Electronics', 'quantity': 1}]",{'method': 'card'},2025-01-03 06:49:40,None,1553.0,ELEVEN11,NaN
1,6ae8c4a71dc54dd2c64a6151,133843,2025-01-04T01:23:47,pending,10997.53,"[{'category': 'Electronics', 'quantity': 1}]",{'method': 'cash_on_delivery'},2025-01-04 10:09:47,None,496.0,NaN,NaN
2,6ae8c4a71dc54dd2c64a615d,126946,2025-01-05T18:26:56,delivered,42176.02,"[{'category': 'Electronics', 'quantity': 3}]",{'method': 'bank_transfer'},2025-01-09 00:40:56,"{'id': 1920, 'name': 'Hassan Mirza', 'city': '...",NaN,NaN,NaN
3,6ae8c4a71dc54dd2c64a615e,162981,2025-01-05T22:55:02,delivered,446.09,"[{'category': 'Books', 'quantity': 1}]",{'method': 'card'},2025-01-11 04:22:02,"{'id': 43, 'name': 'Saad Siddiqui', 'city': 'L...",NaN,NaN,NaN
4,6ae8c4a71dc54dd2c64a6163,294333,2025-01-06T06:28:47,delivered,5856.50,"[{'category': 'Grocery', 'quantity': 1}]",{'method': 'cash_on_delivery'},2025-01-09 11:28:47,None,1176.0,NaN,NaN


In [42]:
def setItemsData(data, name):
    return data[0][name]

In [45]:
orders_df["category"] = orders_df["items"].apply(lambda x: setItemsData(x, "category"))
orders_df["quantity"] = orders_df["items"].apply(lambda x: setItemsData(x, "quantity"))
orders_df.drop(columns=["items"], inplace=True)

/var/folders/41/49dnj5213l1f6btjl3b5rdlh0000gn/T/ipykernel_1716/3064250883.py:3: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  orders_df.drop(columns=["items"], inplace=True)


In [46]:
orders_df.head()

,_id,order_id,order_date,status,amount,payment,updated_at,customer,customer_id,coupon,delivery_notes,category,quantity
0,6ae8c4a71dc54dd2c64a613b,287309,2025-01-02T04:28:40,cancelled,13034.40,{'method': 'card'},2025-01-03 06:49:40,None,1553.0,ELEVEN11,NaN,Electronics,1
1,6ae8c4a71dc54dd2c64a6151,133843,2025-01-04T01:23:47,pending,10997.53,{'method': 'cash_on_delivery'},2025-01-04 10:09:47,None,496.0,NaN,NaN,Electronics,1
2,6ae8c4a71dc54dd2c64a615d,126946,2025-01-05T18:26:56,delivered,42176.02,{'method': 'bank_transfer'},2025-01-09 00:40:56,"{'id': 1920, 'name': 'Hassan Mirza', 'city': '...",NaN,NaN,NaN,Electronics,3
3,6ae8c4a71dc54dd2c64a615e,162981,2025-01-05T22:55:02,delivered,446.09,{'method': 'card'},2025-01-11 04:22:02,"{'id': 43, 'name': 'Saad Siddiqui', 'city': 'L...",NaN,NaN,NaN,Books,1
4,6ae8c4a71dc54dd2c64a6163,294333,2025-01-06T06:28:47,delivered,5856.50,{'method': 'cash_on_delivery'},2025-01-09 11:28:47,None,1176.0,NaN,NaN,Grocery,1


In [56]:
def setPaymentMethod(data):
    return data["method"]

In [58]:
orders_df["payment_method"] = orders_df["payment"].apply(lambda x: setPaymentMethod(x))
orders_df.drop(columns=["payment"], inplace=True)

/var/folders/41/49dnj5213l1f6btjl3b5rdlh0000gn/T/ipykernel_1716/2520242186.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  orders_df.drop(columns=["payment"], inplace=True)


In [59]:
orders_df.head()

,_id,order_id,order_date,status,amount,updated_at,customer,customer_id,coupon,delivery_notes,category,quantity,payment_method
0,6ae8c4a71dc54dd2c64a613b,287309,2025-01-02T04:28:40,cancelled,13034.40,2025-01-03 06:49:40,None,1553.0,ELEVEN11,NaN,Electronics,1,card
1,6ae8c4a71dc54dd2c64a6151,133843,2025-01-04T01:23:47,pending,10997.53,2025-01-04 10:09:47,None,496.0,NaN,NaN,Electronics,1,cash_on_delivery
2,6ae8c4a71dc54dd2c64a615d,126946,2025-01-05T18:26:56,delivered,42176.02,2025-01-09 00:40:56,"{'id': 1920, 'name': 'Hassan Mirza', 'city': '...",NaN,NaN,NaN,Electronics,3,bank_transfer
3,6ae8c4a71dc54dd2c64a615e,162981,2025-01-05T22:55:02,delivered,446.09,2025-01-11 04:22:02,"{'id': 43, 'name': 'Saad Siddiqui', 'city': 'L...",NaN,NaN,NaN,Books,1,card
4,6ae8c4a71dc54dd2c64a6163,294333,2025-01-06T06:28:47,delivered,5856.50,2025-01-09 11:28:47,None,1176.0,NaN,NaN,Grocery,1,cash_on_delivery


In [85]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   _id             5000 non-null   object        
 1   order_id        5000 non-null   int64         
 2   order_date      5000 non-null   str           
 3   status          5000 non-null   str           
 4   amount          5000 non-null   float64       
 5   updated_at      5000 non-null   datetime64[us]
 6   customer        3485 non-null   object        
 7   customer_id     1515 non-null   float64       
 8   coupon          1004 non-null   str           
 9   delivery_notes  449 non-null    str           
 10  category        5000 non-null   str           
 11  quantity        5000 non-null   int64         
 12  payment_method  5000 non-null   str           
dtypes: datetime64[us](1), float64(2), int64(2), object(2), str(6)
memory usage: 507.9+ KB


In [82]:
def setCustomerId(data):
    if data.customer_id >= 0:
        return int(data.customer_id)
    else:
        return data.customer["id"]

In [89]:
orders_df["customer_id"] = orders_df[["customer", "customer_id"]].apply(lambda x: setCustomerId(x), axis=1)
orders_df.drop(columns=["customer"], inplace=True)

/var/folders/41/49dnj5213l1f6btjl3b5rdlh0000gn/T/ipykernel_1716/2020924669.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  orders_df.drop(columns=["customer"], inplace=True)


In [90]:
orders_df.head()

,_id,order_id,order_date,status,amount,updated_at,customer_id,coupon,delivery_notes,category,quantity,payment_method
0,6ae8c4a71dc54dd2c64a613b,287309,2025-01-02T04:28:40,cancelled,13034.40,2025-01-03 06:49:40,1553,ELEVEN11,NaN,Electronics,1,card
1,6ae8c4a71dc54dd2c64a6151,133843,2025-01-04T01:23:47,pending,10997.53,2025-01-04 10:09:47,496,NaN,NaN,Electronics,1,cash_on_delivery
2,6ae8c4a71dc54dd2c64a615d,126946,2025-01-05T18:26:56,delivered,42176.02,2025-01-09 00:40:56,1920,NaN,NaN,Electronics,3,bank_transfer
3,6ae8c4a71dc54dd2c64a615e,162981,2025-01-05T22:55:02,delivered,446.09,2025-01-11 04:22:02,43,NaN,NaN,Books,1,card
4,6ae8c4a71dc54dd2c64a6163,294333,2025-01-06T06:28:47,delivered,5856.50,2025-01-09 11:28:47,1176,NaN,NaN,Grocery,1,cash_on_delivery


In [96]:
orders_df.to_json("raw/mongo_orders.json", orient="records", indent=4, default_handler=str)

/var/folders/41/49dnj5213l1f6btjl3b5rdlh0000gn/T/ipykernel_1716/290467261.py:1: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  orders_df.to_json("raw/mongo_orders.json", orient="records", indent=4, default_handler=str)
